#Chapter 3:
* Building Our Bronze Layer/Project: Retrieval Augmented Generation Chatbot/CH3-01-Creating Embedded Chunks.py

## STEP1: PDFs to binary files

In [0]:
dbutils.widgets.dropdown(name='Reset', defaultValue='True', choices=['True', 'False'], label="Reset: Drop previous table")

# Create a Unity Catalog Volume
spark.sql("CREATE VOLUME IF NOT EXISTS porya_catalog.default.rag_chatbot")
# Sets the file path variable 
volume_file_path = "/Volumes/porya_catalog/default/rag_chatbot"

table_name = "pdf_raw_text"
if bool(dbutils.widgets.get('Reset')):
  sql(f"DROP TABLE IF EXISTS {table_name}")
  sql(f"DROP TABLE IF EXISTS pdf_documentation_text")

In [0]:
# here I want to install all requirements for this project
%pip install -q transformers "unstructured[pdf,docx]" langchain llama-index databricks-vectorsearch pydantic mlflow
%restart_python

In [0]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document, set_global_tokenizer
from transformers import AutoTokenizer
from typing import Iterator
from pyspark.sql.functions import col, udf, length, pandas_udf, explode
import os
import pandas as pd 
from unstructured.partition.auto import partition
from pypdf import PdfReader
import io

# Replacement for mlia_utils.rag_funcs.extract_doc_text
def extract_doc_text(content_bytes):
    reader = PdfReader(io.BytesIO(content_bytes))
    return "\n".join([page.extract_text() or "" for page in reader.pages])

In [0]:
# Reduce the arrow batch size as our PDF can be big in memory
try:
    spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 10)
except Exception as e:
    print(f"Skipping config: {e}")

In [0]:
documents_folder =  f"{volume_file_path}/raw_documents/"
display(dbutils.fs.ls(f"{documents_folder}"))

In [0]:
df = (
        spark.read.format("binaryfile")
        .option("recursiveFileLookup", "true")
        .load('dbfs:'+ documents_folder)
        )

df.write.mode("overwrite").saveAsTable(f"porya_catalog.default.{table_name}")

display(sql(f"SELECT * FROM porya_catalog.default.{table_name} LIMIT 3"))

In [0]:
from unstructured.partition.auto import partition
import re
import io

def extract_doc_text(x : bytes) -> str:
  # Read files and extract the values with unstructured
  sections = partition(file=io.BytesIO(x))
  def clean_section(txt):
    txt = re.sub(r'\n', '', txt)
    return re.sub(r' ?\.', '.', txt)
  # Default split is by section of document
  # concatenate them all together because we want to split by sentence instead.
  return "\n".join([clean_section(s.text) for s in sections]) 

In [0]:
# DBTITLE 1,Basic extracting
with open(f"{documents_folder}/2303.10130.pdf", mode="rb") as pdf:
  doc = extract_doc_text(pdf.read())  
  print(doc)

## STEP2: chunking

In [0]:
@pandas_udf("array<string>")
def read_as_chunk(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    import os
    os.environ["HF_HOME"] = "/tmp/hf_cache"
    os.environ["HF_HUB_CACHE"] = "/tmp/hf_cache"
    os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
    import huggingface_hub.constants
    huggingface_hub.constants.HF_HUB_CACHE = "/tmp/hf_cache"
    #set llama2 as tokenizer
    set_global_tokenizer(
      AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer", cache_dir="/tmp/hf_cache")
    )
    #Sentence splitter from llama_index to split on sentences
    splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50)
    def extract_and_split(b):
      txt = extract_doc_text(b)
      nodes = splitter.get_nodes_from_documents([Document(text=txt)])
      return [n.text for n in nodes]

    for x in batch_iter:
        yield x.apply(extract_and_split)

In [0]:
df_chunks = (df
                .withColumn("content", explode(read_as_chunk("content")))
                .selectExpr('path as pdf_name', 'content')
                )
display(df_chunks)

## STEP3: embedding

In [0]:
def pprint(obj):
  import pprint
  pprint.pprint(obj, compact=True, indent=1, width=100)

In [0]:
from mlflow.deployments import get_deploy_client

# bge-large-en is not available; using gte-large-en instead.
deploy_client = get_deploy_client("databricks")

## NOTE: if you change your embedding model here, make sure you change it in the query step too
embeddings = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": ["What is ChatGPT?"]})
pprint(embeddings)

In [0]:
@pandas_udf("array<float>")
def get_embedding(contents: pd.Series) -> pd.Series:
    import mlflow.deployments
    deploy_client = mlflow.deployments.get_deploy_client("databricks")
    def get_embeddings(batch):
        #Note: this will gracefully fail if an exception is thrown during embedding creation (add try/except if needed) 
        response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": batch})
        return [e['embedding'] for e in response.data]

    # Splitting the contents into batches of 150 items each, since the embedding model takes at most 150 inputs per request.
    max_batch_size = 150
    batches = [contents.iloc[i:i + max_batch_size] for i in range(0, len(contents), max_batch_size)]

    # Process each batch and collect the results
    all_embeddings = []
    for batch in batches:
        all_embeddings += get_embeddings(batch.tolist())

    return pd.Series(all_embeddings)

In [0]:
import pyspark.sql.functions as F

df_chunk_emd = (df_chunks
                .withColumn("embedding", get_embedding("content"))
                .selectExpr('pdf_name', 'content', 'embedding')
                )
display(df_chunk_emd)

df_chunk_emd.write.mode("append").saveAsTable("porya_catalog.default.pdf_documentation_text")